<a href="https://colab.research.google.com/github/CodeBlue0001/BCT-Training-Gen-AI/blob/main/BCT_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Environment

In [ ]:
!pip install -q google-generativeai
!pip install -q gradio
!pip install -q transformers
!pip install -q torch
!pip install -q pypdf
!pip install -q python-docx

In [ ]:
import google.generativeai as genAi
from google.colab import userdata
try:
  KEY=userdata.get('GEMINI_API_KEY')


  genAi.configure(api_key=KEY)

  model=genAi.GenerativeModel("gemini-2.5-flash")

  print(model)
except Exception as e:
  print(e)

In [ ]:
# testing configuration and working or not
res=model.generate_content("What is python?")
res.text

In [ ]:
def ask_gemini(prompt):
    response = model.generate_content(prompt)
    return response.text

In [ ]:
while True:

    question = input("You:")

    if question.lower() == "exit":
        break

    answer = ask_gemini(question)

    print("\nAssistant:")
    print(answer)
    print("-"*60)

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from pypdf import PdfReader

filename = list(uploaded.keys())[0]

reader = PdfReader(filename)

document_text = ""

for page in reader.pages:
    document_text += page.extract_text() + "\n"

print("✅ PDF Loaded Successfully")
print(f"Total Pages: {len(reader.pages)}")

In [ ]:
print(document_text[:10000])

In [ ]:
import nltk
nltk.download()
from nltk import sent_tokenize

nltk.download('punkt_tab')



In [ ]:
sentences = sent_tokenize(document_text)


In [ ]:
words = len(document_text.split())

characters = len(document_text)

print(f"Words : {words}")
print(f"Characters : {characters}")

In [ ]:
def chat_with_document(document, question):

    prompt = f"""
You are an expert AI Research Assistant.

Answer ONLY using the information provided in the document below.

If the answer is not present in the document, clearly say:
"I couldn't find this information in the uploaded document."

---------------- DOCUMENT ----------------

{document}

---------------- USER QUESTION ----------------

{question}

Provide a clear and well-structured answer.
"""

    response = model.generate_content(prompt)

    return response.text

In [ ]:
# test purpose that the engine is running properly or not
while True:
  question = input("Ask something about the document: ")

  answer = chat_with_document(document_text, question)

  print("\n🤖 AI Research Assistant:\n")
  print(answer)
  if question.lower() == "quit":
    break

In [ ]:
from transformers import pipeline

In [ ]:
ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

print("✅ Hugging Face NER Model Loaded")

In [ ]:
entities = ner_pipeline(document_text[:3000])

In [ ]:
for entity in entities:
    print(f"{entity['word']} --> {entity['entity_group']}")

In [ ]:
people = set()
organizations = set()
locations = set()

for entity in entities:

    if entity["entity_group"] == "PER":
        people.add(entity["word"])

    elif entity["entity_group"] == "ORG":
        organizations.add(entity["word"])

    elif entity["entity_group"] == "LOC":
        locations.add(entity["word"])

In [ ]:
print("👤 People")
print(list(people))

print("\n🏢 Organizations")
print(list(organizations))

print("\n📍 Locations")
print(list(locations))

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

In [ ]:
labels = [
    "Research Paper",
    "Resume",
    "Legal Contract",
    "Medical Report",
    "News Article",
    "Business Report",
    "Technical Documentation",
    "Educational Material"
]

result = classifier(
    document_text[:2000],
    candidate_labels=labels
)
result


In [ ]:
print("📄 Predicted Document Type")

for label, score in zip(result["labels"], result["scores"]):
    print(f"{label}: {score:.2%}")

creating the front end


In [ ]:
from pypdf import PdfReader

def extract_pdf_text(file_path):
    reader = PdfReader(file_path)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text

In [ ]:
def ask_document(document, question):

    prompt = f"""
You are an AI Research Assistant.

Document:
{document}

Question:
{question}

Answer only from the document.
If the answer is unavailable, clearly say so.
"""

    response = model.generate_content(prompt)

    return response.text

In [ ]:
import gradio as gr

def process_document_and_ask(pdf_file, question):
    if pdf_file is None:
        return "Please upload a PDF file first."

    file_path = pdf_file.name

    # Ensure the file is saved to a path that pypdf can access
    # Gradio uploads files to a temporary path, which is what we need here.

    document_text = extract_pdf_text(file_path)

    if not document_text:
        return "Could not extract text from the PDF. Please check the file."

    answer = ask_document(document_text, question)
    return answer

# Create the Gradio interface
iface = gr.Interface(
    fn=process_document_and_ask,
    inputs=[
        gr.File(label="Upload your PDF document"),
        gr.Textbox(label="Ask a question about the document:", placeholder="e.g., What is the main purpose of this document?")
    ],
    outputs=gr.Textbox(label="AI Research Assistant:", lines=10),
    title="📄 AI Document Research Assistant",
    description="Upload a PDF document and ask questions about its content. The AI will answer based solely on the provided document."
)

# Launch the interface
iface.launch(debug=True)

In [ ]:
def research_assistant(file_path, question):

    print("Step 1: Reading PDF...")
    document = extract_pdf_text(file_path)

    print("✅ PDF Read Successfully")

    print("Step 2: Classifying document...")
    classification = classify_document(document)

    print("✅ Classification Done")

    print("Step 3: Extracting entities...")
    entities = get_entities(document)

    print("✅ NER Done")

    print("Step 4: Asking Gemini...")
    answer = ask_document(document, question)

    print("✅ Gemini Response Received")

    return classification, entities, answer

In [ ]:
labels = [
    "Research Paper",
    "Resume",
    "Business Report",
    "News Article",
    "Educational Material",
    "Technical Documentation"
]

def classify_document(text):

    result = classifier(
        text[:2000],
        candidate_labels=labels
    )

    return f"{result['labels'][0]} ({result['scores'][0]:.2%})"

In [ ]:
def research_assistant(file, question):

    document = extract_pdf_text(file.name)

    classification = classify_document(document)

    entities = get_entities(document)

    answer = ask_document(document, question)

    return classification, entities, answer

In [ ]:
def get_entities(document_content):
    # Assumes 'ner_pipeline' is globally defined from previous cells
    entities_raw = ner_pipeline(document_content[:3000]) # Limit to 3000 chars as in VAEoZ7Sk8kmm

    people_set = set()
    organizations_set = set()
    locations_set = set()

    for entity in entities_raw:
        if entity["entity_group"] == "PER":
            people_set.add(entity["word"])
        elif entity["entity_group"] == "ORG":
            organizations_set.add(entity["word"])
        elif entity["entity_group"] == "LOC":
            locations_set.add(entity["word"])

    return {"people": list(people_set), "organizations": list(organizations_set), "locations": list(locations_set)}

classification, entities, answer = research_assistant(
    filename,
    "Summarize this document."
)

print(classification)
print()
print(entities)
print()
print(answer)

In [ ]:
with gr.Blocks(theme=gr.themes.Soft(), title="AI Research Assistant Pro") as demo:

    gr.Markdown("# 🤖 AI Research Assistant Pro")
    gr.Markdown("### Powered by Gemini + Hugging Face")

    with gr.Row():

        with gr.Column():

            pdf = gr.File(
                label="Upload PDF",
                file_types=[".pdf"]
            )

            question = gr.Textbox(
                label="Ask a Question",
                placeholder="Example: Summarize this document"
            )

            submit = gr.Button("🚀 Analyze")

        with gr.Column():

            classification = gr.Textbox(
                label="📊 Document Type"
            )

            entities = gr.Textbox(
                label="👥 Named Entities",
                lines=10
            )

            answer = gr.Textbox(
                label="🤖 AI Answer",
                lines=12
            )

    submit.click(
        fn=research_assistant,
        inputs=[pdf, question],
        outputs=[classification, entities, answer]
    )

In [ ]:
demo.launch(share=True, debug=True)

In [ ]:
import gradio as gr

# ============================================================
# Custom CSS
# ============================================================

custom_css = """
body{
    background:#FAFAFA;
}

.gradio-container{
    max-width:1500px !important;
    margin:auto;
    background:#FAFAFA;
    font-family:Inter,Segoe UI,sans-serif;
}

footer{
    display:none !important;
}

/* Header */

.main-title{
    text-align:center;
    font-size:34px;
    font-weight:700;
    color:#2F2F2F;
    margin-top:15px;
}

.sub-title{
    text-align:center;
    color:#777;
    font-size:16px;
    margin-bottom:25px;
}

/* Cards */

.card{
    background:white;
    border-radius:18px;
    border:1px solid #ECECEC;
    padding:18px;
    box-shadow:0 2px 12px rgba(0,0,0,.04);
}

.section-title{
    font-size:18px;
    font-weight:600;
    color:;
    margin-bottom:15px;
}

/* Inputs */

textarea,
input{
    border-radius:12px !important;
    border:1px solid #E5E5E5 !important;
}

textarea:focus,
input:focus{
    border-color:#BFE7D1 !important;
    box-shadow:0 0 0 2px #DDF5E5 !important;
}

/* Button */

button{
    background:#DDF5E5 !important;
    color:#2F5A45 !important;
    border:none !important;
    border-radius:12px !important;
    font-weight:600 !important;
    height:50px !important;
}

button:hover{
    background:#CDEED8 !important;
}

/* Chat */

.chat-answer textarea{
    font-size:15px !important;
    line-height:1.6 !important;
}
"""

# ============================================================
# Theme
# ============================================================

theme = gr.themes.Soft(
    primary_hue="green",
    neutral_hue="gray",
    radius_size="lg"
)

# ============================================================
# UI
# ============================================================

with gr.Blocks(
    theme=theme,
    css=custom_css,
    title="AI Research Assistant"
) as demo:

    gr.HTML("""
    <div class="main-title">
        AI Research Assistant
    </div>

    <div class="sub-title">
        Upload a PDF and ask questions about its contents
    </div>
    """)

    with gr.Row():

        # ====================================================
        # LEFT SIDEBAR
        # ====================================================

        with gr.Column(scale=1):

            with gr.Group(elem_classes="card"):

                gr.HTML(
                    "<div class='section-title'>Upload Document</div>"
                )

                pdf = gr.File(
                    label="Choose PDF",
                    file_types=[".pdf"]
                )

            with gr.Group(elem_classes="card"):

                gr.HTML(
                    "<div class='section-title'>Document Type</div>"
                )

                classification = gr.Textbox(
                    show_label=False,
                    interactive=False,
                    placeholder="Document type..."
                )

            with gr.Group(elem_classes="card"):

                gr.HTML(
                    "<div class='section-title'>Named Entities</div>"
                )

                entities = gr.Textbox(
                    show_label=False,
                    lines=14,
                    interactive=False,
                    placeholder="Detected entities..."
                )

        # ====================================================
        # MAIN CHAT
        # ====================================================

        with gr.Column(scale=3):

            with gr.Group(elem_classes="card"):

                gr.HTML(
                    "<div class='section-title'>AI Response</div>"
                )

                answer = gr.Textbox(
                    show_label=False,
                    lines=24,
                    interactive=False,
                    placeholder="The AI response will appear here...",
                    elem_classes="chat-answer"
                )

            with gr.Group(elem_classes="card"):

                question = gr.Textbox(
                    show_label=False,
                    lines=3,
                    placeholder="Ask anything about your document..."
                )

                submit = gr.Button(
                    "Analyze Document",
                    size="lg"
                )

    submit.click(
        fn=research_assistant,
        inputs=[pdf, question],
        outputs=[
            classification,
            entities,
            answer
        ]
    )

demo.launch()